9번째 코드
8번코드 + 4감정 분류 강화

Train Acc: 94.63% | Val Loss: 1.8127, Val Acc: 81.29%
조기 종료 발동. (Patience: 5)
학습 완료. 저장 위치: /workspace/user1/model_v7_stage2_4class/best_stage2_4class.pth

In [ ]:
import os
os.environ['OPENCV_LOG_LEVEL'] = 'ERROR' # OpenCV 경고 숨김

import json
import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm import tqdm
import random

# ================= 설정 =================
RAW_IMG_BASE = "/workspace/user1/dataset_v2/img/train" 
LBL_BASE = "/workspace/user1/dataset_v2/label_mod/train"
VAL_IMG_BASE = "/workspace/user1/dataset_v2/img/val"
VAL_LBL_BASE = "/workspace/user1/dataset_v2/label_mod/val"
AUG_IMG_BASE = "/workspace/user5/img_augmented/train"

SAVE_DIR = '/workspace/user1/model_v7_stage2_4class'
os.makedirs(SAVE_DIR, exist_ok=True)

SEED = 42
BATCH_SIZE = 64
IMG_SIZE = 224
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# [핵심] Neutral 제외, 4가지 감정만 매핑
STAGE2_MAP = {'anger': 0, 'happy': 1, 'panic': 2, 'sadness': 3}

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"설정 완료. 장치: {DEVICE}")

In [ ]:
class Stage2Dataset(Dataset):
    def __init__(self, raw_img_base, aug_img_base, lbl_base, transform=None, is_train=True):
        self.transform = transform
        self.samples = []
        
        counts = {0: 0, 1: 0, 2: 0, 3: 0}

        print(f"원본 데이터 로딩 중: {lbl_base}")
        if os.path.exists(lbl_base):
            for json_file in tqdm(os.listdir(lbl_base), desc="Loading JSONs", leave=False):
                if not json_file.endswith('.json'): continue
                with open(os.path.join(lbl_base, json_file), 'r', encoding='utf-8') as f:
                    data = json.load(f)
                    for item in data['images']:
                        if item.get('is_box_issue') == 'Yes': continue
                        
                        emotion = item['final_emotion']
                        # Neutral과 Blank는 스킵 (이진 분류에서 이미 처리됨)
                        if emotion in ['neutral', 'blank']: continue
                        
                        label = STAGE2_MAP[emotion]
                        counts[label] += 1

                        self.samples.append({
                            'source': 'raw',
                            'img_path': os.path.join(raw_img_base, emotion, item['filename']),
                            'bbox': item['bbox'],
                            'label': label
                        })

        if is_train and os.path.exists(aug_img_base):
            print(f"증강 데이터 로딩 중: {aug_img_base}")
            for folder_name in STAGE2_MAP.keys():
                src_folder = os.path.join(aug_img_base, folder_name)
                if not os.path.isdir(src_folder): continue
                
                label = STAGE2_MAP[folder_name]
                file_list = [f for f in os.listdir(src_folder) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
                
                for filename in file_list:
                    counts[label] += 1
                    self.samples.append({
                        'source': 'aug',
                        'img_path': os.path.join(src_folder, filename),
                        'label': label
                    })

        print(f"로딩 완료. (Anger: {counts[0]}, Happy: {counts[1]}, Panic: {counts[2]}, Sadness: {counts[3]})")

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        image = cv2.imread(item['img_path'])
        
        if image is None: return self.__getitem__(random.randint(0, len(self)-1))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        if item['source'] == 'raw':
            box = item['bbox']
            x1, y1, x2, y2 = max(0, box['minX']), max(0, box['minY']), box['maxX'], box['maxY']
            if x2 <= x1 or y2 <= y1: return self.__getitem__(random.randint(0, len(self)-1))
            image = image[y1:y2, x1:x2]
            
        if self.transform:
            image = self.transform(image=image)['image']
            
        return image, torch.tensor(item['label'], dtype=torch.long)

In [ ]:
train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=15, p=0.4),
    A.ColorJitter(brightness=0.1, contrast=0.1, p=0.3),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

train_dataset = Stage2Dataset(RAW_IMG_BASE, AUG_IMG_BASE, LBL_BASE, transform=train_transform, is_train=True)
val_dataset = Stage2Dataset(VAL_IMG_BASE, AUG_IMG_BASE, VAL_LBL_BASE, transform=val_transform, is_train=False)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

print("DataLoader 설정 완료.")

In [ ]:
def get_stage2_model():
    # num_classes를 4로 설정
    model = timm.create_model('efficientnet_b2', pretrained=True, num_classes=4)
    fer_url = "https://github.com/sb-ai-lab/EmotiEffLib/raw/main/models/affectnet_emotions/enet_b2_8.pt"
    
    try:
        loaded = torch.hub.load_state_dict_from_url(fer_url, map_location='cpu', weights_only=False)
        if hasattr(loaded, 'state_dict'): loaded = loaded.state_dict()
        filtered_dict = {k: v for k, v in loaded.items() if 'classifier' not in k and 'head' not in k and 'fc' not in k}
        model.load_state_dict(filtered_dict, strict=False)
        print("가중치 주입 성공.")
    except Exception as e:
        print(f"가중치 주입 실패. 기본 가중치 사용. ({e})")
    return model

model = get_stage2_model().to(DEVICE)

# Anger 데이터가 다른 3개 감정에 비해 다소 적으므로 약간의 가중치 부여
# Anger(0): 1.5, Happy(1): 1.0, Panic(2): 1.0, Sadness(3): 1.0
class_weights = torch.tensor([1.5, 1.0, 1.0, 1.0]).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.05)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=2)

NUM_EPOCHS = 30
PATIENCE = 5
best_val_loss = float('inf')
trigger_times = 0

SAVE_PATH = os.path.join(SAVE_DIR, 'best_stage2_4class.pth')
scaler = torch.amp.GradScaler('cuda')

print("[Phase 2] 4분류(Anger, Happy, Panic, Sadness) 학습 시작...")

for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss, correct_train, total_train = 0.0, 0, 0

    train_pbar = tqdm(train_loader, desc=f"Ep [{epoch+1}/{NUM_EPOCHS}] Train")
    for images, labels in train_pbar:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()

        with torch.autocast(device_type='cuda', dtype=torch.float16):
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()
        train_pbar.set_postfix({'loss': f"{loss.item():.4f}"})

    train_loss = running_loss / len(train_dataset)
    train_acc = correct_train / total_train * 100

    model.eval()
    val_loss, correct_val, total_val = 0.0, 0, 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            with torch.autocast(device_type='cuda', dtype=torch.float16):
                outputs = model(images)
                loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()

    val_loss = val_loss / len(val_dataset)
    val_acc = correct_val / total_val * 100
    
    scheduler.step(val_loss)

    print(f"Train Acc: {train_acc:.2f}% | Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), SAVE_PATH)
        print(f"  -> 모델 저장됨 (Val Loss: {val_loss:.4f})")
        trigger_times = 0
    else:
        trigger_times += 1
        if trigger_times >= PATIENCE:
            print(f"조기 종료 발동. (Patience: {PATIENCE})")
            break

print(f"학습 완료. 저장 위치: {SAVE_PATH}")

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# 2단계 4분류 모델 로드
model = get_stage2_model().to(DEVICE)
model.load_state_dict(torch.load('/workspace/user1/model_v7_stage2_4class/best_stage2_4class.pth'))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, lbls in val_loader:
        imgs = imgs.to(DEVICE)
        with torch.autocast(device_type='cuda', dtype=torch.float16):
            out = model(imgs)
        all_preds.extend(torch.max(out, 1)[1].cpu().numpy())
        all_labels.extend(lbls.numpy())

# 혼동 행렬 시각화
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges', 
            xticklabels=['Anger', 'Happy', 'Panic', 'Sadness'],
            yticklabels=['Anger', 'Happy', 'Panic', 'Sadness'])
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Stage 2 (4-Class) Confusion Matrix')
plt.show()